# Task 3: Retail Data Analysis

This notebook explores the Sample Superstore dataset to identify sales, profitability and customer-segment patterns.

**Objectives**

- Clean and validate the dataset
- Examine sales and profitability
- Compare regions, categories and sub-categories
- Investigate customer segments
- Identify loss-making categories, sub-categories and markets
- Present evidence-based business recommendations

## 1. Import libraries and load data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

DATA_PATH = Path("data/sample_superstore.csv")
df = pd.read_csv(DATA_PATH)

df.head()

## 2. Clean and validate the dataset

In [ ]:
# Standardise column names and remove exact duplicate records.
df.columns = df.columns.str.strip()
duplicate_rows = df.duplicated().sum()
df = df.drop_duplicates().copy()

numeric_columns = ["Sales", "Quantity", "Discount", "Profit"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["Profit Margin (%)"] = (
    df["Profit"].div(df["Sales"]).mul(100).where(df["Sales"].ne(0))
)

quality_summary = pd.DataFrame({
    "Metric": ["Rows after cleaning", "Columns", "Exact duplicates removed", "Missing values"],
    "Value": [len(df), len(df.columns), duplicate_rows, int(df.isna().sum().sum())],
})
quality_summary

In [ ]:
df.info()
df.isna().sum().sort_values(ascending=False)

## 3. Overall sales and profitability

In [ ]:
kpis = pd.DataFrame({
    "Metric": ["Total sales", "Total profit", "Profit margin (%)", "Average sales per record", "Loss-making records"],
    "Value": [
        df["Sales"].sum(),
        df["Profit"].sum(),
        df["Profit"].sum() / df["Sales"].sum() * 100,
        df["Sales"].mean(),
        (df["Profit"] < 0).sum(),
    ],
})
kpis

## 4. Customer-segment performance

In [ ]:
segment_summary = (
    df.groupby("Segment", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Records=("Sales", "size"))
      .assign(**{"Profit Margin (%)": lambda table: table["Profit"].div(table["Sales"]).mul(100)})
      .sort_values("Profit", ascending=False)
)
segment_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=segment_summary, x="Segment", y="Sales", ax=axes[0])
axes[0].set_title("Sales by Customer Segment")
axes[0].set_xlabel("")

sns.barplot(data=segment_summary, x="Segment", y="Profit", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Profit by Customer Segment")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

## 5. Regional and category performance

In [ ]:
region_summary = (
    df.groupby("Region", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
      .assign(**{"Profit Margin (%)": lambda table: table["Profit"].div(table["Sales"]).mul(100)})
      .sort_values("Profit", ascending=False)
)

category_summary = (
    df.groupby("Category", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
      .sort_values("Profit", ascending=False)
)

display(region_summary)
display(category_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=region_summary, x="Region", y="Profit", ax=axes[0])
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Profit by Region")

sns.barplot(data=category_summary, x="Category", y="Sales", ax=axes[1])
axes[1].set_title("Sales by Category")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 6. Sub-category profitability and discount impact

In [ ]:
subcategory_summary = (
    df.groupby(["Category", "Sub-Category"], as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Average_Discount=("Discount", "mean"))
      .sort_values("Profit")
)

loss_makers = subcategory_summary.query("Profit < 0").copy()
display(loss_makers)

discount_summary = (
    df.groupby("Discount", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Records=("Sales", "size"))
      .sort_values("Discount")
)
discount_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=subcategory_summary.sort_values("Profit").head(10),
    x="Profit", y="Sub-Category", hue="Category", dodge=False, ax=axes[0]
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Ten Least Profitable Sub-Categories")
axes[0].legend(title="Category", bbox_to_anchor=(1, 1))

sns.lineplot(data=discount_summary, x="Discount", y="Profit", marker="o", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Profit by Discount Level")

plt.tight_layout()
plt.show()

## 7. Loss-making markets

The dataset does not include product names, customer IDs or order dates. This analysis therefore identifies loss-making **regions, categories and sub-categories**, rather than individual products or customers.

In [ ]:
loss_making_markets = (
    df[df["Profit"] < 0]
      .groupby(["Region", "Category", "Sub-Category"], as_index=False)
      .agg(Loss=("Profit", "sum"), Sales=("Sales", "sum"), Average_Discount=("Discount", "mean"), Records=("Profit", "size"))
      .sort_values("Loss")
)
loss_making_markets.head(15)

## 8. Business recommendations

1. **Review the sub-categories shown in `loss_makers`.** Prioritise the categories with the largest negative total profit.
2. **Set discount guardrails.** Use the discount chart to review levels where profit drops below zero; discounts should be tested against margin, not sales volume alone.
3. **Target regional improvement.** Investigate loss-making region/category combinations in `loss_making_markets` before expanding promotions there.
4. **Protect profitable customer segments.** Use `segment_summary` to focus retention and cross-sell activity on segments that combine strong sales with healthy profit margins.
5. **Collect additional fields for deeper analysis.** Product name, customer ID, order ID and order date would enable product-level, customer-level and time-based recommendations.

> These findings are descriptive. They show patterns in the dataset, not proof that a particular discount or region causes profit changes.